
# 1. Business Understanding

## 1.1 Project Background

The project focuses on the Brazilian Olist e-commerce marketplace.

The objective is to understand the relationship between delivery
performance, customer satisfaction and sales.

The analysis uses historical Olist e-commerce data from
January 2017 to August 2018.

## 1.2 Business Problem

Olist has data distributed across multiple source files.
There is no single trusted view of delivery performance,
customer satisfaction and sales.

The relationship between late deliveries and customer
review scores needs to be investigated.

## 1.3 Main Research Question

What is the relationship between delivery performance
and customer satisfaction in the Olist marketplace?

## 1.4 Supporting Business Questions

1. How many orders were delivered on time and late?
2. How long do orders take to reach customers?
3. How many days are orders delayed?
4. Is delivery delay related to customer review scores?
5. Do late deliveries have more poor reviews?
6. Does delivery performance change over time?
7. Does delivery performance vary by geographical area?
8. Is order value different between late and on-time deliveries?

## 1.5 Analysis Objectives

- Measure delivery performance.
- Identify delivery delays and patterns.
- Analyse customer review scores.
- Examine the relationship between delivery performance
  and customer satisfaction.
- Compare sales indicators across delivery groups.
- Develop useful business insights.

## 1.6 Project Scope

Dataset: Brazilian Olist e-commerce dataset.

Analysis period: January 2017 to August 2018.

Tools: Python, MySQL and Power BI.

Source data: Nine original CSV files.



## 1.7 Initial KPI Framework

| KPI | Definition |
|---|---|
| Total Orders | Number of unique orders |
| Delivered Orders | Number of successfully delivered orders |
| On-Time Delivery Rate | On-time delivered orders / total delivered orders × 100 |
| Late Delivery Rate | Late delivered orders / total delivered orders × 100 |
| Average Delivery Time | Average days between purchase and actual delivery |
| Average Delay Days | Average number of days late for delayed orders |
| Average Review Score | Average customer review score |
| Poor Review Rate | Orders with 1–2 star reviews / reviewed orders × 100 |
| Average Order Value (AOV) | Total order value / number of orders |
| Gross Merchandise Value (GMV) | Total item price, excluding freight |

*Note: These are initial KPI definitions. Final calculations will be validated after reviewing the data dictionary, data quality and table relationships.*


# 2. Data Loading

## 2.1 Objective

The objective of this stage is to load the nine original Olist
CSV files into Python using Pandas.

The datasets will be checked for file availability, successful
loading, row counts, column counts and basic structure.

The original source files will remain unchanged.

In [34]:
from pathlib import Path

# Find the Olist project folder
current_folder = Path.cwd().resolve()

PROJECT_ROOT = None

for folder in [current_folder, *current_folder.parents]:
    if (folder / "02_data_acquisition" / "raw_data").is_dir():
        PROJECT_ROOT = folder
        break

# Fallback to the known project location
if PROJECT_ROOT is None:
    possible_path = (
        Path.home()
        / "Documents/University/BI_BigData/Olist_Project"
    )

    if (possible_path / "02_data_acquisition" / "raw_data").is_dir():
        PROJECT_ROOT = possible_path

if PROJECT_ROOT is None:
    raise FileNotFoundError("Olist project folder not found.")

# Define the data location
DATA_DIR = PROJECT_ROOT / "02_data_acquisition" / "raw_data"

print("Project location:", PROJECT_ROOT)
print("Data location:", DATA_DIR)
print("Data folder exists:", DATA_DIR.exists())

Project location: /Users/tarun/Documents/University/BI_BigData/Olist_Project
Data location: /Users/tarun/Documents/University/BI_BigData/Olist_Project/02_data_acquisition/raw_data
Data folder exists: True


## 2.3 Check the available CSV files

In [35]:
csv_files = sorted(DATA_DIR.glob("*.csv"))

print("Number of CSV files:", len(csv_files))

for file in csv_files:
    print(file.name)

Number of CSV files: 9
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


## 2.4 Load the nine datasets into Python

In this step, we use Pandas to load all nine CSV files
into memory as separate DataFrames.

Each DataFrame represents one source table.
The original CSV files will remain unchanged.

In [36]:

import pandas as pd
import numpy as np

# Define the nine source files
DATA_FILES = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

# Create an empty dictionary
data = {}

# Load each CSV file
for dataset_name, filename in DATA_FILES.items():

    file_path = DATA_DIR / filename

    data[dataset_name] = pd.read_csv(
        file_path,
        low_memory=False
    )

    print(f"Successfully loaded: {dataset_name}")

print("\nTotal datasets loaded:", len(data))


Successfully loaded: customers
Successfully loaded: geolocation
Successfully loaded: order_items
Successfully loaded: order_payments
Successfully loaded: order_reviews
Successfully loaded: orders
Successfully loaded: products
Successfully loaded: sellers
Successfully loaded: category_translation

Total datasets loaded: 9


## 2.5 Dataset Overview

This section summarises the number of records and
columns in each loaded dataset.

In [37]:
dataset_summary = []

for name, df in data.items():

    dataset_summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Memory (MB)": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2), 2
        )
    })

dataset_summary_df = pd.DataFrame(dataset_summary)

display(dataset_summary_df)

,Dataset,Rows,Columns,Memory (MB)
0,customers,99441,5,26.59
1,geolocation,1000163,5,129.38
2,order_items,112650,7,35.99
3,order_payments,103886,5,16.23
4,order_reviews,99224,7,39.12
5,orders,99441,8,52.94
6,products,32951,9,6.30
7,sellers,3095,4,0.59
8,category_translation,71,2,0.01


## 2.6 Initial Inspection of the Orders Dataset

The orders dataset contains order information, order status,
purchase timestamps and delivery timestamps.

It will be a key source for analysing delivery performance.

In [38]:
# Display the first five records
display(data["orders"].head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [39]:
# Display the column names
print("Columns in the orders dataset:")

for column in data["orders"].columns:
    print(column)

Columns in the orders dataset:
order_id
customer_id
order_status
order_purchase_timestamp
order_approved_at
order_delivered_carrier_date
order_delivered_customer_date
order_estimated_delivery_date


In [40]:
# Check the data types
data["orders"].dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

## 2.7 Initial Order Status Overview

This section examines the different order statuses
recorded in the Olist orders dataset.

In [41]:
# Count orders by status
order_status_summary = (
    data["orders"]["order_status"]
    .value_counts(dropna=False)
)

display(order_status_summary)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [42]:
# Create reports folder
REPORT_DIR = PROJECT_ROOT / "04_eda" / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save dataset overview
summary_file = REPORT_DIR / "02_data_loading_summary.csv"

dataset_summary_df.to_csv(summary_file, index=False)

print("Dataset summary successfully saved!")
print("File location:", summary_file)

Dataset summary successfully saved!
File location: /Users/tarun/Documents/University/BI_BigData/Olist_Project/04_eda/reports/02_data_loading_summary.csv
